In [ ]:
import numpy as np

# represent the velocity components, as well as weights. In order (rest, up, down, left, right, top-right, bottom-right, bottom-left, top-left)

c_x = np.array([0, 0, 0, -1, 1, 1, 1, -1, -1], dtype=np.int16)
c_y = np.array([0, 1, -1, 0, 0, 1, -1, -1, 1], dtype=np.int16)
weights = np.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36], dtype=np.float64)

In [ ]:
# now define matrices for 2D slice Ny, Nx dimensions. define matrices/tensors for rho, u, and f_i
Nx = 400
Ny = 100

# base density is one
rho = np.ones((Ny, Nx))
# fluid at rest before beginning
u = np.zeros((2, Ny, Nx))
# since density = 1, and velocity = 0, the f_i = weights. 
f_i = weights.reshape((9, 1, 1)) * np.ones((Ny, Nx))

# part 2: thermal coupling
# start by initialising T to be ones
T = np.ones((Ny, Nx))
# make g_i with the same logic as how we initialised f_i
g_i = weights.reshape((9, 1, 1)) * T

In [ ]:
# define centre of cylinder + radius
cx = Nx // 4
cy = (Ny // 2) - 2
R = 13

# coordinates
x_coord = np.arange(Nx)
y_coord = np.arange(Ny)

# create matrices X and Y, where each element possesses its own x-coord or y-coord for X and Y respectively
X, Y = np.meshgrid(x_coord, y_coord)

# create cylinder array with True to represent in cylinder and False to represent elsewhere
cylinder = (X - cx)**2 + (Y - cy)**2 <= R**2

In [ ]:
# function to calculate f_i for equilibrium at every time step
def get_equilibrium(rho, u):
    u_x = u[0, :, :]
    u_y = u[1, :, :]

    # compute (u_x^2 + u_y^2) term as it is same for all directions
    const = u_x**2 + u_y**2
    # create base tensor for equlibrium state
    feq_i_base = np.zeros((9, Ny, Nx))
    # now calculate equlibrium distribution in each direction using a for loop
    for i in range(9):
        direct_term = c_x[i] * u_x + c_y[i] * u_y
        feq_i = weights[i] * rho * (1 + 3 * direct_term + 4.5 * direct_term**2 - 1.5*const)
        feq_i_base[i, :, :] = feq_i
    return feq_i_base

In [ ]:
# now begin to code the BGK operator, define relaxation time as well
Nt = 4000
tau = 0.7

# will now use an 1D array to represent the index of directions opposite to the direction currently selected
# needed to halp simulate motion of fluid if comes in contact with the cylinder
opposite = [0, 2, 1, 4, 3, 7, 8, 5, 6]

# define inlet velocity in mach number terms
u_inlet = 0.04

# temperature of disk
T_hot = 2.0

# change T so that pixels inside cylinder are at temperature T_hot
T[cylinder] = T_hot

alpha = 0.86
for time_step in range(Nt):
    f_i = f_i - (1/tau) * (f_i - get_equilibrium(rho, u))
    g_i = g_i - (1/alpha) * (g_i - get_equilibrium(T, u))
    # now simulating streaming of the particles
    for i in range(9):
        f_i[i, :, :] = np.roll(f_i[i, :, :], (c_y[i], c_x[i]), axis=(0,1))
        g_i[i, :, :] = np.roll(g_i[i, :, :], (c_y[i], c_x[i]), axis=(0, 1))

    # now index position of cylinder
    boundary = f_i[:, cylinder]
    # flip directions of the f_i as the fluid flows in opposite direction after coming in contact with cylinder
    f_i[:, cylinder] = boundary[opposite]

    # make last column of f_i = second last of f_i, same for g_i
    f_i[:, :, -1] = f_i[:, :, -2]
    g_i[:, :, -1] = g_i[:, :, -2]

    # calculate rho by summing f_i
    rho = np.sum(f_i, axis=0)

    # update velocity, by summing all momentums in all directions and then dividing by density
    # initialise the u_x and u_y to 0
    u[0, :, :] = 0
    u[1, :, :] = 0

    # multiply each probability distribution for each direction by vector in that direction
    for i in range(9):
        u[0, :, :] += f_i[i, :, :] * c_x[i]
        u[1, :, :] += f_i[i, :, :] * c_y[i]

    # divide by rho to get final speed, add microscopic number to prevent NaN appearing
    u[0, :, :] /= (rho + 1e-9)
    u[1, :, :] /= (rho + 1e-9)

    # now calculate T
    T = np.sum(g_i, axis=0)
    
    # set u_x to be equal to u_inlet for first column
    u[0, :, 0] = u_inlet
    # set u_y to be 0
    u[1, :, 0] = 0
    
    # set rho to be 1 throughout first column, same for T
    rho[:, 0] = 1
    T[:, 0] = 1

    # set velocity in cylinder to be 0. this prevents simulation breaking
    u[0, cylinder] = 0
    u[1, cylinder] = 0

    # set f_i of first column to be equal to what is the equilibrium, same for g_i
    f_i[:, :, 0] = get_equilibrium(rho, u)[:, :, 0]
    g_i[:, :, 0] = get_equilibrium(T, u)[:, :, 0]

    # force thermal distribution at boundary to equal equilibrium
    g_i[:, cylinder] = get_equilibrium(T, u)[:, cylinder]

    # reapply boundary condition so disk doesnt cool
    T[cylinder] = T_hot
    

In [ ]:
import matplotlib.pyplot as plt

# we want 2D heatmap of scalar speed values
u_2d = np.sqrt(u[0, :, :]**2 + u[1, :, :]**2)

plt.imshow(u_2d, cmap='jet')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(T, cmap='inferno')
plt.show()

In [ ]:
# calculate bernoulli pressure
p = rho / 3

# create 1 pixel layer around the disk to find the boundary particles
master_mask = np.full((Ny, Nx), False)

# shift shape in one of each direction and then use or operator to merge. ignore 0 as no direction
for i in range(1, 9):
    shifted_shape = np.roll(cylinder, (c_y[i], c_x[i]), axis=(0,1))
    master_mask |= shifted_shape

# subtract inside to just get outer boundary
envelope = master_mask &~ cylinder

# isolate only particles surrounding the circle
coords_surround = np.argwhere(envelope)

# find the coordinates of the particles next to these that are in the cylinder
T_infty = 1.0
nablaT = 0
drag_force = 0
for array in coords_surround:
    p_local = p[array[0], array[1]]
    for i in range(1, 9):
        check_coord = (array[0] + c_y[i], array[1] + c_x[i])
        if i > 4 and cylinder[check_coord] == True:
            nablaT += (T_hot - T[array[0], array[1]])/np.sqrt(2)
            drag_force += p_local * c_x[i]
        elif i < 5 and cylinder[check_coord] == True:
            nablaT += (T_hot - T[array[0], array[1]])
            drag_force += p_local * c_x[i]

# calculate average h
average_grad = nablaT / len(coords_surround)
h = average_grad / (T_hot - T_infty)

print(f'Calculated h: {h}')
print(f'Calculated F_drag: {drag_force}')

In [ ]:
import os
import pandas as pd
import glob


# set up directory etc
dataset_dir = "dataset_geometries"
csv_file = "dataset_metrics.csv"
os.makedirs(dataset_dir, exist_ok=True)

# continue from where it left off
existing_files = glob.glob(os.path.join(dataset_dir, "geometry_*.npy"))
start_id = len(existing_files) + 1

# initialise list to act as ledger
ledger = []
total_target = 500

# start with id 1
for design_id in range(start_id, total_target + 1):
    # ----------------- random shape generation model -----------------------------
    # generate angles of points and distances from centre on shape
    num_points = 10
    angles = np.linspace(0, 2*np.pi, num_points, endpoint=False)
    centre_disp = np.random.uniform(low=5, high=25, size=(num_points,))
    
    # now calculate exact points of each point on polygon
    x_coords = cx + centre_disp * np.cos(angles)
    y_coords = cy + centre_disp * np.sin(angles)
    
    # now colour in inside of shape
    from matplotlib.path import Path
    # expects in format (x, y) so do as such
    vertices = np.column_stack((x_coords, y_coords))
    duct_path = Path(vertices)
    
    # use our X and Y from previously to create a flat grid
    grid_points = np.column_stack((X.flatten(), Y.flatten()))
    
    # now check if points are inside polygon
    inside_polygon = duct_path.contains_points(grid_points)
    # previous line returns 1D array so reshape back into 2D array
    cylinder = inside_polygon.reshape((Ny, Nx))
    # -------------------- reset simulation -----------------------------------
    rho = np.ones((Ny, Nx))
    u = np.zeros((2, Ny, Nx))
    T = np.ones((Ny, Nx))
    
    u[0, :, :] = u_inlet
    T[cylinder] = 2.0
    
    f_i = get_equilibrium(rho, u)
    g_i = get_equilibrium(T, u)
    # -------------------- start of LBM simulation ------------------------------
    # now begin to code the BGK operator, define relaxation time as well
    Nt = 4000
    tau = 0.7
    
    # will now use an 1D array to represent the index of directions opposite to the direction currently selected
    # needed to halp simulate motion of fluid if comes in contact with the cylinder
    opposite = [0, 2, 1, 4, 3, 7, 8, 5, 6]
    
    # define inlet velocity in mach number terms
    u_inlet = 0.04
    
    # temperature of disk
    T_hot = 2.0
    
    # change T so that pixels inside cylinder are at temperature T_hot
    T[cylinder] = T_hot
    
    alpha = 0.86
    for time_step in range(Nt):
        f_i = f_i - (1/tau) * (f_i - get_equilibrium(rho, u))
        g_i = g_i - (1/alpha) * (g_i - get_equilibrium(T, u))
        # now simulating streaming of the particles
        for i in range(9):
            f_i[i, :, :] = np.roll(f_i[i, :, :], (c_y[i], c_x[i]), axis=(0,1))
            g_i[i, :, :] = np.roll(g_i[i, :, :], (c_y[i], c_x[i]), axis=(0, 1))
    
        # now index position of cylinder
        boundary = f_i[:, cylinder]
        # flip directions of the f_i as the fluid flows in opposite direction after coming in contact with cylinder
        f_i[:, cylinder] = boundary[opposite]
    
        # make last column of f_i = second last of f_i, same for g_i
        f_i[:, :, -1] = f_i[:, :, -2]
        g_i[:, :, -1] = g_i[:, :, -2]
    
        # calculate rho by summing f_i
        rho = np.sum(f_i, axis=0)
    
        # update velocity, by summing all momentums in all directions and then dividing by density
        # initialise the u_x and u_y to 0
        u[0, :, :] = 0
        u[1, :, :] = 0

        # multiply each probability distribution for each direction by vector in that direction
        for i in range(9):
            u[0, :, :] += f_i[i, :, :] * c_x[i]
            u[1, :, :] += f_i[i, :, :] * c_y[i]
    
        # divide by rho to get final speed, add microscopic number to prevent NaN appearing
        u[0, :, :] /= (rho + 1e-9)
        u[1, :, :] /= (rho + 1e-9)
    
        # now calculate T
        T = np.sum(g_i, axis=0)
        
        # set u_x to be equal to u_inlet for first column
        u[0, :, 0] = u_inlet
        # set u_y to be 0
        u[1, :, 0] = 0
        
        # set rho to be 1 throughout first column, same for T
        rho[:, 0] = 1
        T[:, 0] = 1
    
        # set velocity in cylinder to be 0. this prevents simulation breaking
        u[0, cylinder] = 0
        u[1, cylinder] = 0
    
        # set f_i of first column to be equal to what is the equilibrium, same for g_i
        f_i[:, :, 0] = get_equilibrium(rho, u)[:, :, 0]
        g_i[:, :, 0] = get_equilibrium(T, u)[:, :, 0]
    
        # force thermal distribution at boundary to equal equilibrium
        g_i[:, cylinder] = get_equilibrium(T, u)[:, cylinder]
    
        # reapply boundary condition so disk doesnt cool
        T[cylinder] = T_hot
        pass

    # ------------------------ end of LBM simulation -----------------------------------
    # save matrix to disk to protect RAM
    filename = f"geometry_{design_id:03d}.npy"
    filepath = os.path.join(dataset_dir, filename)
    np.save(filepath, cylinder)

    # ------------------------- extraction metrics ---------------------------------
    # calculate bernoulli pressure
    p = rho / 3
    
    # create 1 pixel layer around the disk to find the boundary particles
    master_mask = np.full((Ny, Nx), False)
    
    # shift shape in one of each direction and then use or operator to merge. ignore 0 as no direction
    for i in range(1, 9):
        shifted_shape = np.roll(cylinder, (c_y[i], c_x[i]), axis=(0,1))
        master_mask |= shifted_shape
    
    # subtract inside to just get outer boundary
    envelope = master_mask &~ cylinder
    
    # isolate only particles surrounding the circle
    coords_surround = np.argwhere(envelope)
    
    # find the coordinates of the particles next to these that are in the cylinder
    T_infty = 1.0
    nablaT = 0
    drag_force = 0
    for array in coords_surround:
        p_local = p[array[0], array[1]]
        for i in range(1, 9):
            check_coord = (array[0] + c_y[i], array[1] + c_x[i])
            if i > 4 and cylinder[check_coord] == True:
                nablaT += (T_hot - T[array[0], array[1]])/np.sqrt(2)
                drag_force += p_local * c_x[i]
            elif i < 5 and cylinder[check_coord] == True:
                nablaT += (T_hot - T[array[0], array[1]])
                drag_force += p_local * c_x[i]
    
    # calculate average h
    average_grad = nablaT / len(coords_surround)
    h = average_grad / (T_hot - T_infty)
    # ------------------ end of extraction metrics -------------------------

    # -------------------------- appending stuff ---------------------------------
    # convert the single iterations metrics into a DataFrame
    row_df = pd.DataFrame([[design_id, drag_force, h]])
    
    # append directly to the CSV on your hard drive
    # (header=False prevents it from writing column names 500 times)
    row_df.to_csv(csv_file, mode='a', header=False, index=False)

    print(f"Design {design_id:03d} Complete | Drag: {drag_force:.4f} | h: {h:.4f}")

df = pd.DataFrame(ledger)
df.to_csv(csv_file, index=False)
print("\nDataset generation complete. Metrics saved to CSV.")

In [2]:
!pip install torch

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 1.5 MB/s eta 0:01:21
   ---------------------------------------- 0.8/124.1 MB 1.4 MB/s eta 0:01:31
   ---------------------------------------- 1.0/124.1 MB 1.5 MB/s eta 0:01:24
   ---------------------------------------- 1.3/124.1 MB 1.2 MB/s eta 0:01:39
    --------------------------------------- 1.6/124.1 MB 1.3 MB/s eta 0:01:37
    --------------------------------------- 2.1/124.1 MB 1.4 MB/s eta 0:01:25
    --------------------------------------- 2.6/124.1 MB 1.6 MB/s eta 0:01:18
   - -------------------------------------- 3.1/124.1 MB 1.7 MB/s eta 0:01:12
   - -------------------------------------- 3.7/124.1 MB 1.7 MB/s eta 0:01:10
   - -------------------------------------- 4.2/124.1 MB 1.8 MB/s eta 0:01:08
   - --

In [12]:
# now coding the CNN
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import os

class BrakeDuctDataset(Dataset):
    def __init__(self, csv_file, root_dir):
        # load master ledger into memory
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        # get the 'design_id' from the dataframe at row 'idx'
        self.data_frame.iloc[idx]['design_id']
    
        # reconstruct the exact filename (e.g., 'geometry_042.npy') 
        row = self.data_frame.iloc[idx]
        design_id = int(row['design_id'])
        filename = f"geometry_{design_id:03d}.npy"

        file_path = os.path.join(self.root_dir, filename)
        
        # load the matrix using np.load()
        numpy_matrix = np.load(file_path)
        
        # convert the NumPy array to a PyTorch FloatTensor.  
        # add .unsqueeze(0) to the end to turn the [100, 400] matrix into a [1, 100, 400] tensor, will only be accepted in [Channel, Height, Width].
        image_tensor = torch.tensor(numpy_matrix, dtype=torch.float32).unsqueeze(0)
        
        # extract the 'drag_force' and 'heat_transfer_h' from the dataframe at row 'idx'
        drag = row['drag_force']
        h = row['heat_transfer_h']
        
        # pack both metrics into a single 1D PyTorch tensor
        labels_tensor = torch.tensor([drag, h], dtype=torch.float32)
        
        # return the two tensors
        return image_tensor, labels_tensor

In [13]:
# initialise data pipeline
my_dataset = BrakeDuctDataset(csv_file="dataset_metrics.csv", root_dir="dataset_geometries")

# pull the very first example
sample_image, sample_labels = my_dataset[0]

print("Image Tensor Shape:", sample_image.shape)
print("Labels Tensor:", sample_labels)

Image Tensor Shape: torch.Size([1, 100, 400])
Labels Tensor: tensor([0.0949, 0.1192])


In [14]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

# cnn architecture
class SurrogateCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(38400, 512)
        self.fc2 = nn.Linear(512, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# data prep and batching
# initialize the dataset 
dataset = BrakeDuctDataset(csv_file="dataset_metrics.csv", root_dir="dataset_geometries")

# reserve ~15% of the data to test if the model actually learned the physics
test_size = int(0.15 * len(dataset))
train_size = len(dataset) - test_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# DataLoader feeds the matrices into the CNN in batches of 16
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# training engine setup
# automatically use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SurrogateCNN().to(device)

# MSE for continuous regression
criterion = nn.MSELoss()
# Adam optimizer handles the gradient descent steps
optimizer = optim.Adam(model.parameters(), lr=0.001)

# master training loop
num_epochs = 15

print(f"Starting neural network training on {device}...\n")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        # move data to the active device
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()            
        predictions = model(images)       
        loss = criterion(predictions, labels) 
        loss.backward()                   
        optimizer.step()                  
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | Mean Squared Error: {avg_loss:.6f}")

print("\nPhase 4 Complete: Surrogate Model Successfully Trained.")

Starting neural network training on cpu...

Epoch [01/15] | Mean Squared Error: 0.103008
Epoch [02/15] | Mean Squared Error: 0.003932
Epoch [03/15] | Mean Squared Error: 0.003338
Epoch [04/15] | Mean Squared Error: 0.003023
Epoch [05/15] | Mean Squared Error: 0.002679
Epoch [06/15] | Mean Squared Error: 0.002455
Epoch [07/15] | Mean Squared Error: 0.002627
Epoch [08/15] | Mean Squared Error: 0.002249
Epoch [09/15] | Mean Squared Error: 0.002276
Epoch [10/15] | Mean Squared Error: 0.002239
Epoch [11/15] | Mean Squared Error: 0.002162
Epoch [12/15] | Mean Squared Error: 0.002044
Epoch [13/15] | Mean Squared Error: 0.002037
Epoch [14/15] | Mean Squared Error: 0.002060
Epoch [15/15] | Mean Squared Error: 0.001861

Phase 4 Complete: Surrogate Model Successfully Trained.


In [15]:
# lock the model for testing (disables dropout and batch norm variations)
model.eval()

total_drag_error = 0.0
total_h_error = 0.0
sample_count = 0

print("CFD Ground Truth vs. CNN Surrogate Prediction\n" + "-"*50)

# disable the autograd engine to save memory and speed up inference
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        # get the network's predictions
        predictions = model(images)
        
        # calculate absolute errors for this batch
        drag_error = torch.abs(predictions[:, 0] - labels[:, 0])
        h_error = torch.abs(predictions[:, 1] - labels[:, 1])
        
        total_drag_error += torch.sum(drag_error).item()
        total_h_error += torch.sum(h_error).item()
        sample_count += labels.size(0)

        # print the first 5 examples from the very first batch
        if sample_count <= 16: 
            for i in range(min(5, labels.size(0))):
                actual_drag = labels[i, 0].item()
                pred_drag = predictions[i, 0].item()
                
                actual_h = labels[i, 1].item()
                pred_h = predictions[i, 1].item()
                
                print(f"Test Geometry {i+1}:")
                print(f"  Drag -> Actual CFD: {actual_drag:.4f} | AI Predicted: {pred_drag:.4f}")
                print(f"  h    -> Actual CFD: {actual_h:.4f} | AI Predicted: {pred_h:.4f}\n")

# calculate and print the global accuracy metrics
mean_drag_error = total_drag_error / sample_count
mean_h_error = total_h_error / sample_count

print("-" * 50)
print("GLOBAL TEST SET METRICS (Mean Absolute Error):")
print(f"Average Drag Error: ±{mean_drag_error:.4f}")
print(f"Average h Error:    ±{mean_h_error:.4f}")

CFD Ground Truth vs. CNN Surrogate Prediction
--------------------------------------------------
Test Geometry 1:
  Drag -> Actual CFD: 0.2920 | AI Predicted: 0.2300
  h    -> Actual CFD: 0.1066 | AI Predicted: 0.1028

Test Geometry 2:
  Drag -> Actual CFD: 0.3618 | AI Predicted: 0.2809
  h    -> Actual CFD: 0.1032 | AI Predicted: 0.1024

Test Geometry 3:
  Drag -> Actual CFD: 0.2662 | AI Predicted: 0.2566
  h    -> Actual CFD: 0.0931 | AI Predicted: 0.0983

Test Geometry 4:
  Drag -> Actual CFD: 0.3750 | AI Predicted: 0.3023
  h    -> Actual CFD: 0.1020 | AI Predicted: 0.1011

Test Geometry 5:
  Drag -> Actual CFD: 0.2828 | AI Predicted: 0.2738
  h    -> Actual CFD: 0.1031 | AI Predicted: 0.1060

--------------------------------------------------
GLOBAL TEST SET METRICS (Mean Absolute Error):
Average Drag Error: ±0.0537
Average h Error:    ±0.0068
